# Step 04B — Checkpoint Provenance and Native CV Re-evaluation

**Purpose:** Re-evaluate **existing** per-fold checkpoints for YOLO11s, RT-DETR-L, Faster R-CNN and RetinaNet on their 14-image held-out validation fold. Compare actual replayed mAP values with (A) the present Drive CSV files and (B) outputs retained in the originally executed research notebook. No training, no annotation changes, no checkpoint overwriting.

**Important:** This is a **checkpoint/provenance and native-evaluator reconciliation**. It is **NOT** yet the cross-framework unified AP control (that requires exporting and re-scoring all four models with the same evaluator and protocol). The current source labels remain source-defined despite the findings of independent dental experts.

- Use a **T4 GPU** runtime; run all cells sequentially.
- The output is a ZIP file in `MyDrive/Pediatric_Caries_AI/reviewer_revision/Step_04B_...`.
- The notebook fails safely on missing/invalid reference data and preserves all original files.
- The historical reference table is transcribed from the saved outputs of `Pediatric_Caries_Detection_Project (2).ipynb`; the current reference comes from **Step 04A** inventory and is embedded below.
- A metric mismatch should be documented, not repaired by choosing whichever number is higher.

In [ ]:
# Install only evaluation dependencies. Do NOT reinstall torch / torchvision.
%pip -q install "ultralytics==8.4.138" "torchmetrics>=1.3,<2" pycocotools

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,hashlib,json,time,gc,zipfile,platform,traceback
from datetime import datetime,timezone
import pandas as pd
import numpy as np
import torch,torchvision
from PIL import Image
from scipy.stats import friedmanchisquare
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics import YOLO, RTDETR
import ultralytics,torchmetrics

ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
CV=ROOT/'cross_validation'
assert CV.is_dir(), f'Project missing: {CV}'
assert torch.cuda.is_available(), 'T4 GPU is required. Runtime > Change runtime type > T4 GPU'
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'Step_04B_Checkpoint_Replay_{RUN_ID}'
OUT.mkdir(parents=True, exist_ok=False)
DEV='cuda:0'
print('Output:',OUT)
print('GPU:',torch.cuda.get_device_name(0))
print('Versions:',{'python':sys.version.split()[0], 'torch':torch.__version__,'torchvision':torchvision.__version__,
                  'ultralytics':ultralytics.__version__,'torchmetrics':torchmetrics.__version__})
with open(OUT/'environment.json','w') as f:
 json.dump({'utc':RUN_ID,'python':sys.version,'platform':platform.platform(),
    'torch':torch.__version__,'torchvision':torchvision.__version__,
    'ultralytics':ultralytics.__version__,'torchmetrics':torchmetrics.__version__,
    'gpu':torch.cuda.get_device_name(0)},f,indent=2)

In [ ]:
# Reference tables: current 04A files versus historical executed-notebook outputs.
CURRENT_REFERENCE = [{'family': 'Faster R-CNN', 'fold': 1, 'mAP50': 0.2572338581085205, 'mAP50_95': 0.1323672086000442}, {'family': 'Faster R-CNN', 'fold': 2, 'mAP50': 0.2894095182418823, 'mAP50_95': 0.1212050914764404}, {'family': 'Faster R-CNN', 'fold': 3, 'mAP50': 0.2136123031377792, 'mAP50_95': 0.1121814101934433}, {'family': 'Faster R-CNN', 'fold': 4, 'mAP50': 0.3474820852279663, 'mAP50_95': 0.165413960814476}, {'family': 'Faster R-CNN', 'fold': 5, 'mAP50': 0.1529041230678558, 'mAP50_95': 0.0677801668643951}, {'family': 'RT-DETR-L', 'fold': 1, 'mAP50': 0.3102861230362855, 'mAP50_95': 0.1703365633134867}, {'family': 'RT-DETR-L', 'fold': 2, 'mAP50': 0.4195915534798811, 'mAP50_95': 0.2269464310083449}, {'family': 'RT-DETR-L', 'fold': 3, 'mAP50': 0.1911203258334149, 'mAP50_95': 0.1204239676901199}, {'family': 'RT-DETR-L', 'fold': 4, 'mAP50': 0.2946930535113553, 'mAP50_95': 0.1412126038135144}, {'family': 'RT-DETR-L', 'fold': 5, 'mAP50': 0.2922965769720776, 'mAP50_95': 0.152568249484162}, {'family': 'RetinaNet', 'fold': 1, 'mAP50': 0.0718571841716766, 'mAP50_95': 0.0420792065560817}, {'family': 'RetinaNet', 'fold': 2, 'mAP50': 0.3154663741588592, 'mAP50_95': 0.1536078006029129}, {'family': 'RetinaNet', 'fold': 3, 'mAP50': 0.0, 'mAP50_95': 0.0}, {'family': 'RetinaNet', 'fold': 4, 'mAP50': 0.2811713218688965, 'mAP50_95': 0.1147802025079727}, {'family': 'RetinaNet', 'fold': 5, 'mAP50': 0.1746227443218231, 'mAP50_95': 0.0755548924207687}, {'family': 'YOLO11s', 'fold': 1, 'mAP50': 0.309315369356687, 'mAP50_95': 0.1855169355692881}, {'family': 'YOLO11s', 'fold': 2, 'mAP50': 0.5296247330427906, 'mAP50_95': 0.2622745199093148}, {'family': 'YOLO11s', 'fold': 3, 'mAP50': 0.2620254940615761, 'mAP50_95': 0.1328469171235302}, {'family': 'YOLO11s', 'fold': 4, 'mAP50': 0.4691166719005339, 'mAP50_95': 0.2493837249667272}, {'family': 'YOLO11s', 'fold': 5, 'mAP50': 0.1999358647622687, 'mAP50_95': 0.1086185998974849}]
HISTORICAL_NOTEBOOK = {'YOLO11s': [[0.309315, 0.185517], [0.529625, 0.262275], [0.262025, 0.132847], [0.469117, 0.249384], [0.199936, 0.108619]], 'RT-DETR-L': [[0.310286, 0.170337], [0.419592, 0.226946], [0.19112, 0.120424], [0.294693, 0.141213], [0.188727, 0.09555]], 'Faster R-CNN': [[0.249069, 0.141778], [0.282479, 0.141351], [0.250011, 0.13126], [0.273838, 0.137086], [0.244315, 0.134039]], 'RetinaNet': [[0.17761, 0.095206], [0.115001, 0.058394], [0.065381, 0.030047], [0.146535, 0.057462], [0.171662, 0.097494]]}

cur=pd.DataFrame(CURRENT_REFERENCE).rename(columns={'mAP50':'drive_mAP50','mAP50_95':'drive_mAP50_95'})
hrows=[]
for fam,arr in HISTORICAL_NOTEBOOK.items():
    for fold,(a,b) in enumerate(arr,1):
        hrows.append({'family':fam,'fold':fold,'historical_mAP50':a,'historical_mAP50_95':b})
historical=pd.DataFrame(hrows)
refs=cur.merge(historical,on=['family','fold'],validate='one_to_one')
assert len(refs)==20
refs['drive_minus_historical_mAP50']=refs.drive_mAP50-refs.historical_mAP50
refs['drive_minus_historical_mAP50_95']=refs.drive_mAP50_95-refs.historical_mAP50_95
refs.to_csv(OUT/'reference_values_and_differences.csv',index=False)
print(refs.to_string(index=False,float_format=lambda n:f'{n:.6f}'))

In [ ]:
# Checkpoints and fold GT sanity. Hash only 20 principal "best.pt" checkpoints (no duplicate  -2/-3 runs).
from torchvision.models.detection import fasterrcnn_resnet50_fpn,retinanet_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

FAMILIES=['YOLO11s','RT-DETR-L','Faster R-CNN','RetinaNet']
EXPECTED_GT=[50,48,31,49,28]

def checkpoint_path(family,fold):
    if family=='YOLO11s': return CV/f'models/YOLO11s/fold_{fold}/weights/best.pt'
    if family=='RT-DETR-L': return CV/f'models/RTDETR/fold_{fold}/weights/best.pt'
    if family=='Faster R-CNN': return CV/f'models/FasterRCNN/fold_{fold}/best.pt'
    if family=='RetinaNet': return CV/f'models/RetinaNet/fold_{fold}/best.pt'
    raise ValueError(family)

def sha256(path):
    hh=hashlib.sha256()
    with open(path,'rb') as fh:
        for chunk in iter(lambda:fh.read(4*1024*1024),b''): hh.update(chunk)
    return hh.hexdigest()

def fold_dataset_paths(fold):
    base=CV/f'fold_{fold}'
    ims=sorted(p for p in (base/'images/val').iterdir() if p.suffix.lower() in ['.png','.jpg','.jpeg','.bmp'])
    assert len(ims)==14, f'Fold {fold}: need 14 validation images; found {len(ims)}'
    lab=base/'labels/val'
    num=0
    for im in ims:
        lbl=lab/(im.stem+'.txt')
        assert lbl.exists(),f'Missing empty/positive label file: {lbl}'
        txt=lbl.read_text().strip()
        if txt:
            for line in txt.splitlines():
                parts=line.split()
                assert len(parts)==5 and int(float(parts[0]))==0 and all(np.isfinite([float(v) for v in parts])),f'Invalid YOLO label: {lbl}'
                num+=1
    assert num==EXPECTED_GT[fold-1],f'Fold {fold}: expected {EXPECTED_GT[fold-1]} caries boxes, got {num}. STOP.'
    return ims,lab,num

inventory=[]
for fold in range(1,6):
    ims,lab,num=fold_dataset_paths(fold)
    for family in FAMILIES:
        p=checkpoint_path(family,fold)
        assert p.is_file(),f'Missing checkpoint: {p}'
        row={'family':family,'fold':fold,'checkpoint_relative_path':str(p.relative_to(ROOT)),
             'bytes':p.stat().st_size,'sha256':sha256(p),
             'val_images':len(ims),'val_boxes':num}
        inventory.append(row)
        print(family,fold,p.name,round(p.stat().st_size/1024/1024,1),'MiB',row['sha256'][:16])
inv=pd.DataFrame(inventory)
inv.to_csv(OUT/'primary_checkpoint_sha256_and_fold_counts.csv',index=False)

# Hash current CV result CSVs, record duplicate archive matches when available.
map_csv={'YOLO11s':'YOLO11s_results.csv','RT-DETR-L':'RTDETR_results.csv',
    'Faster R-CNN':'FasterRCNN_results.csv','RetinaNet':'RetinaNet_results.csv'}
rows=[]
for fam,filename in map_csv.items():
    src=CV/filename
    archived=ROOT/'reviewer_revision/Step_03_Archived_Statistical_Evidence'/('ARCHIVED_'+filename)
    rows.append({'family':fam,'drive_csv_sha256':sha256(src),
      'archive_csv_sha256':sha256(archived) if archived.exists() else None,
      'same_bytes_if_archive_exists':src.read_bytes()==archived.read_bytes() if archived.exists() else None})
pd.DataFrame(rows).to_csv(OUT/'cv_csv_archival_identity.csv',index=False)

In [ ]:
# Original-framework native evaluator for torchvision detectors.
# Mirrors the study's PIL->tensor, YOLO xywh normalized -> pixel xyxy, class id 1, COCO mAP workflow.

def load_validation_sample(imgpath,label_dir):
    im=Image.open(imgpath).convert('RGB')
    W,H=im.size
    x=torchvision.transforms.functional.to_tensor(im)
    txt=(label_dir/(imgpath.stem+'.txt')).read_text().strip()
    boxes=[]
    if txt:
        for line in txt.splitlines():
            cl,xc,yc,bw,bh=map(float,line.split())
            xc*=W; yc*=H; bw*=W; bh*=H
            box=[xc-bw/2,yc-bh/2,xc+bw/2,yc+bh/2]
            assert box[0]>=-0.1 and box[1]>=-0.1 and box[2]<=W+0.1 and box[3]<=H+0.1
            assert box[2]>box[0] and box[3]>box[1]
            boxes.append(box)
    b=torch.tensor(boxes,dtype=torch.float32).reshape(-1,4)
    t={'boxes':b,'labels':torch.ones((len(boxes),),dtype=torch.int64)}
    return x,t

def build_torchvision_detector(family):
    if family=='Faster R-CNN':
        m=fasterrcnn_resnet50_fpn(weights=None,weights_backbone=None)
        nf=m.roi_heads.box_predictor.cls_score.in_features
        m.roi_heads.box_predictor=FastRCNNPredictor(nf,2)
    elif family=='RetinaNet':
        m=retinanet_resnet50_fpn(weights=None,weights_backbone=None)
        na=m.head.classification_head.num_anchors
        inc=m.backbone.out_channels
        m.head.classification_head=RetinaNetClassificationHead(inc,na,2)
    else:raise ValueError(family)
    return m

@torch.inference_mode()
def replay_torchvision(family,fold,weights):
    ims,label_dir,_=fold_dataset_paths(fold)
    m=build_torchvision_detector(family)
    state=torch.load(str(weights),map_location='cpu',weights_only=True)
    if isinstance(state,dict) and 'state_dict' in state: state=state['state_dict']
    m.load_state_dict(state,strict=True)
    del state
    m=m.to(DEV).eval()
    metric=MeanAveragePrecision(box_format='xyxy',iou_type='bbox',backend='pycocotools')
    predrows=[]
    for im in ims:
        image,target=load_validation_sample(im,label_dir)
        out=m([image.to(DEV)])[0]
        prediction={k:out[k].detach().cpu() for k in ['boxes','scores','labels']}
        metric.update([prediction],[target])
        for bb,ss,ll in zip(prediction['boxes'].tolist(),prediction['scores'].tolist(),prediction['labels'].tolist()):
            predrows.append({'family':family,'fold':fold,'image':im.name,
               'x1':bb[0],'y1':bb[1],'x2':bb[2],'y2':bb[3],'score':ss,'class_original':ll,'class_normalized':0 if ll==1 else -1})
    r=metric.compute()
    del m,metric
    gc.collect()
    torch.cuda.empty_cache()
    return float(r['map_50']),float(r['map']),predrows

def replay_ultralytics(family,fold,weights):
    cls=YOLO if family=='YOLO11s' else RTDETR
    obj=cls(str(weights))
    r=obj.val(data=str(CV/f'fold_{fold}'/'data.yaml'),split='val',imgsz=1024,
         conf=0.001,iou=0.70,max_det=300,device=0,batch=1,workers=0,
         half=False,plots=False,verbose=False,save=False,
         project=str(OUT/'ultralytics_native_val'),name=family.replace(' ','_')+f'_fold{fold}',exist_ok=False)
    v=(float(r.box.map50),float(r.box.map))
    del obj,r
    gc.collect();torch.cuda.empty_cache()
    return v[0],v[1],[]

In [ ]:
# Do not train. Re-evaluate all 20 saved best checkpoints against the corresponding val fold.
# Save an incremental CSV after EVERY fold, so completed work survives any runtime interruption.
rows=[]; failures=[]
for family in FAMILIES:
  for fold in range(1,6):
    p=checkpoint_path(family,fold)
    t=time.time()
    try:
      if family in ['YOLO11s','RT-DETR-L']:
        m50,m95,pr=replay_ultralytics(family,fold,p)
      else:
        m50,m95,pr=replay_torchvision(family,fold,p)
      row={'family':family,'fold':fold,'replayed_mAP50':m50,'replayed_mAP50_95':m95,
           'checkpoint_sha256':inv.loc[(inv.family==family)&(inv.fold==fold),'sha256'].iloc[0],
           'seconds':round(time.time()-t,1),'status':'OK','error':''}
      if pr:
        pd.DataFrame(pr).to_csv(OUT/(f"predictions_{family.replace(' ','_')}_fold{fold}.csv"),index=False)
      print('OK',family,fold,'mAP50=',round(m50,6),'mAP50_95=',round(m95,6))
    except Exception as exc:
      row={'family':family,'fold':fold,'replayed_mAP50':np.nan,'replayed_mAP50_95':np.nan,
           'checkpoint_sha256':inv.loc[(inv.family==family)&(inv.fold==fold),'sha256'].iloc[0],
           'seconds':round(time.time()-t,1),'status':'ERROR','error':repr(exc)}
      failures.append((family,fold,repr(exc)))
      print('ERROR',family,fold,repr(exc))
      traceback.print_exc(limit=2)
    rows.append(row)
    pd.DataFrame(rows).to_csv(OUT/'native_replay_per_fold.csv',index=False)
    gc.collect();torch.cuda.empty_cache()
print('Completed:',len(rows),'success:',sum(r['status']=='OK' for r in rows),'errors:',len(failures))

In [ ]:
# Comparisons and exploratory statistics: every result remains labeled by source.
replay=pd.read_csv(OUT/'native_replay_per_fold.csv')
combined=refs.merge(replay,on=['family','fold'],how='left',validate='one_to_one')
for metric in ['mAP50','mAP50_95']:
    combined[f'abs_delta_vs_drive_{metric}']=(combined[f'replayed_{metric}']-combined[f'drive_{metric}']).abs()
    combined[f'abs_delta_vs_notebook_{metric}']=(combined[f'replayed_{metric}']-combined[f'historical_{metric}']).abs()
combined.to_csv(OUT/'replay_vs_both_reference_versions.csv',index=False)
print(combined[['family','fold','drive_mAP50','historical_mAP50','replayed_mAP50',
   'abs_delta_vs_drive_mAP50','abs_delta_vs_notebook_mAP50']].to_string(index=False))

summ=[]
for fam,g in combined.groupby('family'):
    for met in ['mAP50','mAP50_95']:
        for source in ['drive','historical','replayed']:
            v=g[f'{source}_{met}'].dropna()
            summ.append({'family':fam,'metric':met,'source':source,'n_folds':len(v),
                         'mean':v.mean(),'sd':v.std(ddof=1) if len(v)>1 else np.nan})
pd.DataFrame(summ).to_csv(OUT/'source_version_mean_sd.csv',index=False)

stats=[]
for source in ['drive','historical','replayed']:
 for metric in ['mAP50','mAP50_95']:
    wide=combined.pivot(index='fold',columns='family',values=f'{source}_{metric}').reindex(columns=FAMILIES)
    if wide.notna().all().all():
        test=friedmanchisquare(*[wide[c].to_numpy() for c in FAMILIES])
        stats.append({'source':source,'metric':metric,'friedman_chi2':float(test.statistic),'friedman_p_asymptotic':float(test.pvalue),
          'note':'Exploratory: five nonindependent CV training folds; do not interpret as independent replicate experiments'})
pd.DataFrame(stats).to_csv(OUT/'friedman_by_source_version.csv',index=False)

summary={'completed_successfully':int((replay.status=='OK').sum()),'total_folds':20,
         'match_to_drive_under_0_005':int((combined.abs_delta_vs_drive_mAP50<.005).sum()),
         'match_to_notebook_under_0_005':int((combined.abs_delta_vs_notebook_mAP50<.005).sum()),
         'interpretation':'Native evaluator replay is provenance evidence, not a unified framework evaluation; do not automatically replace historical manuscript figures.'}
with open(OUT/'run_summary.json','w') as f:json.dump(summary,f,indent=2)
print('Summary',json.dumps(summary,indent=2))

zip_dest=OUT.parent/(OUT.name+'_Results.zip')
with zipfile.ZipFile(zip_dest,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
  for p in sorted(OUT.rglob('*')):
    if p.is_file(): z.write(p,p.relative_to(OUT))
print('RESULT ZIP:',zip_dest)
print('Upload this ZIP to ChatGPT. If a fold failed, the ZIP includes the traceback status and successful folds.')